## 01. 데이터 탐색 (EDA)

매매·전월세 테이블의 구조, 규모, 주소 형식을 확인합니다.

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가 (notebooks/ -> part10_price_volume/ -> projects/ -> 프로젝트 루트)
PROJECT_ROOT = str(Path.cwd().resolve().parents[2])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from projects.part10_price_volume.dashboard.db import execute_query

### 1. 테이블 행 수 확인

In [ ]:
# 주요 테이블 행 수 확인
tables = ['매매', '전월세']
for t in tables:
    df = execute_query(f"SELECT COUNT(*) as cnt FROM {t}")
    print(f"{t}: {df.iloc[0]['cnt']:,}건")

### 2. 매매 테이블 구조

In [ ]:
# 매매 테이블 컬럼 구조
df = execute_query("DESCRIBE 매매")
print(df[["column_name", "column_type"]].to_string(index=False))

In [ ]:
# 매매 샘플 5건
df = execute_query("SELECT * FROM 매매 LIMIT 5")
display(df)

### 3. 전월세 테이블 — 전월세구분 분포

In [ ]:
# 전세 vs 월세 건수
df = execute_query("""
    SELECT 전월세구분, COUNT(*) AS cnt
    FROM 전월세
    GROUP BY 전월세구분
    ORDER BY cnt DESC
""")
print(df)

### 4. 시군구 컬럼의 주소 형식 분석

시군구 컬럼은 `시도 시군구 읍면동` 형태로 하나의 문자열에 들어 있다.
단어 수에 따라 주소 구조가 다양하다.

In [ ]:
# 거래량 상위 5개 시군구값
df = execute_query("""
    SELECT
        시군구,
        COUNT(*) AS cnt
    FROM 매매
    WHERE 시군구 IS NOT NULL
    GROUP BY 시군구
    ORDER BY cnt DESC
    LIMIT 5
""")
print(df)

In [ ]:
# 단어 수별 분포 (공백 기준 split)
df = execute_query("""
    SELECT
        len(regexp_split_to_array(regexp_replace(시군구, '\\s+', ' ', 'g'), ' ')) AS word_count,
        COUNT(*) AS cnt
    FROM 매매
    WHERE 시군구 IS NOT NULL
    GROUP BY word_count
    ORDER BY word_count
""")
display(df)

In [ ]:
# 세종특별자치시 더블스페이스 확인
df = execute_query("""
    SELECT DISTINCT 시군구
    FROM 매매
    WHERE 시군구 LIKE '세종%'
    LIMIT 5
""")
for val in df['시군구']:
    print(repr(val))

### 5. 거래유형 및 해제사유발생일 (필터 대상 컬럼)

In [ ]:
# 거래유형 분포
df = execute_query("""
    SELECT 거래유형, COUNT(*) AS cnt
    FROM 매매
    GROUP BY 거래유형
    ORDER BY cnt DESC
""")
display(df)

In [ ]:
# 해제사유발생일 NULL이 아닌 값의 종류 (취소 거래 확인)
df = execute_query("""
    SELECT
        CASE
            WHEN 해제사유발생일 IS NULL THEN 'NULL'
            WHEN 해제사유발생일 IN ('None', '-', '') THEN 해제사유발생일
            ELSE '날짜값'
        END AS 구분,
        COUNT(*) AS cnt
    FROM 매매
    GROUP BY 구분
    ORDER BY cnt DESC
""")
display(df)

### 6. 계약년월 범위 확인

In [ ]:
from projects.part10_price_volume.dashboard.preprocessing import get_date_range

min_ym, max_ym = get_date_range()
print(f"계약년월 범위: {min_ym} ~ {max_ym}")
print(f"  시작: {min_ym // 100}년 {min_ym % 100}월")
print(f"  종료: {max_ym // 100}년 {max_ym % 100}월")